In [0]:
%python
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

CAT = "dev_insurance"
OPS = f"/Volumes/{CAT}/ops"

def apply_dq(df, source):
    """Adds _dq_issues (array of 'severity:rule_name') and _is_error (any error-severity issue)."""
    rules = df.sparkSession.table(f"{CAT}.ops.dq_rules").where(f"source = '{source}' AND active").collect()
    issue_cols = []
    for r in rules:
        col = f"_r_{r.rule_id}"
        df = df.withColumn(col,
              F.when(F.coalesce(F.expr(r.expression), F.lit(False)), F.lit(None).cast("string"))
               .otherwise(F.lit(f"{r.severity}:{r.rule_name}")))
        issue_cols.append(col)
    return (df.withColumn("_dq_issues", F.array_compact(F.array(*[F.col(c) for c in issue_cols])))
              .drop(*issue_cols)
              .withColumn("_is_error", F.exists("_dq_issues", lambda x: x.startswith("error:"))))